[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_8_deep_sarsa.ipynb)

<div style="text-align:center">
    <h1>
        Deep SARSA
    </h1>
</div>

<br><br>

<div style="text-align:center">

In this notebook, we extend the SARSA algorithm to use function approximators (Neural Networks). The resulting algorithm is known as Deep SARSA.
</div>


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_stats, seed_everything, plot_cost_to_go, plot_max_q


## Import the necessary software libraries:

In [ ]:
import random
import copy
import gym
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch import nn as nn
from torch.optim import AdamW
from tqdm import tqdm

## Create and prepare the environment

### Create the environment

### Getting the environment ready for PyTorch

A network speaks **tensors**, but gym returns NumPy arrays. We need a wrapper that converts every observation, reward, and done-flag into a PyTorch tensor and adds a leading **batch dimension** (`1 x state_dims`), since PyTorch layers expect a batch.

> **TODO (you):** First create and seed `MountainCar-v0` and read off `state_dims` / `num_actions`. Then write a `PreprocessEnv(gym.Wrapper)` whose `reset` and `step` return tensors (hint: `torch.from_numpy(...).unsqueeze(0).float()`), and wrap the env with it.

### Prepare the environment to work with PyTorch

## Create the Q-Network and policy

<br><br>

### Create the Q-Network: $\hat q(s,a| \theta)$

### Approximating Q-values with a neural network

A table cannot cover MountainCar's infinitely many continuous states, so we use a **function approximator**: a small network that maps the raw state to one Q-value **per action**. Two pieces:

1. **The Q-network `q(s,a|theta)`** - the weights we train. Being smooth, it gives nearby states similar Q-values (the generalization a table lacks).
2. **A frozen target network `q(s,a|theta_targ)`** - a copy used to compute the learning target, so we are not chasing a target that moves every update.

> **TODO (you):** Build the `q_network` as an `nn.Sequential` MLP (input `state_dims`, output `num_actions`, with hidden `ReLU` layers), then make `target_q_network` a frozen `copy.deepcopy(...).eval()`, and write an epsilon-greedy `policy` that reads action values from the network.

### Create the target Q-Network: $\hat q(s, a|\theta_{targ})$

### Create the $\epsilon$-greedy policy: $\pi(s)$

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

## Create the Experience Replay buffer

<br>
<div style="text-align:center">
    <p>A simple buffer that stores transitions of arbitrary values, adapted from
    <a href="https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html#training">this source.</a></p>
</div>


### Experience Replay: don't learn from steps in order

Consecutive transitions are highly **correlated**, which destabilises gradient descent. A **replay buffer** stores many past transitions so we can train on a **random minibatch** each update - decorrelating the data and reusing each experience many times.

> **TODO (you):** Implement a `ReplayMemory` class with `insert`, `sample`, and `can_sample` methods (a circular buffer is a clean approach).

## Implement the algorithm

</br></br>

### Deep SARSA: SARSA with a neural network in place of the table

Replace the Q-table with the neural network. The update is still SARSA - only the pieces are 'deep':

- **`Q(s,a)`** comes from a forward pass, not a table lookup; use `gather` to select the Q-value of the action actually taken in each batch row.
- **The SARSA target** is `r + gamma * Q(s', a')`, where `a'` is chosen by the *same* epsilon-greedy policy (on-policy - that is what makes it SARSA, not Q-Learning's max).
- **The target network** supplies `Q(s', a')` and is refreshed only every few episodes for stability.
- **`~done`** zeroes the future term on the final step of an episode.
- **Learning** is regression: minimise MSE between predicted `Q(s,a)` and the target, then `zero_grad -> backward -> step`.

> **TODO (you):** Write `deep_sarsa(...)`: create an `AdamW` optimizer and a `ReplayMemory`, loop over episodes collecting transitions, and once you can sample a minibatch, compute the SARSA target with the target network and take one gradient step. Periodically copy the weights into the target network. Then train it in the next cell.

## Show results

### Plot execution stats

In [ ]:
plot_stats(stats)

### Plot the cost to go: $ - \max_a \hat q(s,a|\theta)$

In [ ]:
plot_cost_to_go(env, q_network, xlabel='Car Position', ylabel='Velocity')

### Show resulting policy: $\pi(s)$

In [ ]:
plot_max_q(env, q_network, xlabel='Car Position', ylabel='Velocity',
           action_labels=['Back', 'Do nothing', 'Forward'])

### Test the resulting agent

In [ ]:
test_agent(env, policy, episodes=2)

## Resources

[[1] Deep Reinforcement Learning with Experience Replay Based on SARSA](https://www.researchgate.net/publication/313803199_Deep_reinforcement_learning_with_experience_replay_based_on_SARSA)